# YOLO26s baseline — AI City 3 lớp

Notebook này chuẩn bị dữ liệu và tạo lượt baseline YOLO26s; chưa có SCConv và không chạy train trong workspace hiện tại.

## Cách chạy trên Colab
1. Đưa 5 phần dữ liệu vào Drive và chia sẻ CẢ thư mục AI_City_5_parts lẫn AI_City_project cho tài khoản Colab nhận. Tài khoản nhận cần thêm shortcut của mỗi thư mục vào My Drive; notebook remount Drive để dùng đúng tài khoản và tự dò folder theo tên hoặc cấu trúc. Nếu để trong Shared Drive, notebook cũng tự dò các thư mục cùng tên.
2. Ở cell cấu hình, chỉ sửa SOURCE_ROOT/PROJECT_ROOT nếu shortcut có tên hoặc vị trí khác; có thể thêm đường dẫn vào *_FALLBACKS. Không bỏ qua cảnh báo nếu không tìm thấy project dùng chung: cần thấy last.pt và metadata cũ trước khi resume. Chọn GPU rồi chạy tuần tự từ đầu; sau restart runtime, chạy lại từ cell cấu hình.
3. Dataset chuẩn hóa, config, checkpoint, args, CSV và log đều nằm dưới PROJECT_ROOT trên Drive. Nếu chuyển tài khoản, đăng nhập đúng tài khoản đã được chia sẻ, mount Drive, chạy lại notebook từ đầu; cell cấu hình sẽ in đường dẫn thực tế đã nhận diện. Cache dataset chấp nhận SOURCE_ROOT khác khi fingerprint dữ liệu giống nhau.
4. Nếu phiên trước bị ngắt, chạy cell train lại: notebook đọc last.pt, in số epoch đã hoàn tất và resume cùng run đến tổng 100 epoch. Nếu file khóa còn tồn tại, chỉ xóa lock dưới PROJECT_ROOT/locks sau khi chắc chắn phiên cũ đã dừng. Không mở cùng run ở hai tài khoản đồng thời.
5. Không dùng validation để gọi là test. Notebook này không đọc test. Dự đoán validation dùng ngưỡng cố định chỉ để rà lỗi; không tối ưu ngưỡng trên test.

Workspace observation: `81_100 - Minh` currently contains 4,318 images: 3,510 image+label copies matching the first four parts and 808 unique images from sequences 081-083 and 085-100. Its local README/YAML metadata points to `61_80 - Khoi`; the notebook warns about this and uses the class names from each data.yaml. It collapses a copy only after image SHA256 and source-label checksum match exactly; other annotation conflicts stop the cell.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from google.colab import drive

# Remount để dùng đúng tài khoản sau khi chuyển; chỉ chạy cell này khi phiên train trước đã dừng.
drive.mount("/content/drive", force_remount=True)

# Đường dẫn chuẩn. Khi chia sẻ, thêm shortcut vào My Drive với đúng tên này.
SOURCE_ROOT = Path("/content/drive/MyDrive/AI_City_5_parts")
PROJECT_ROOT = Path("/content/drive/MyDrive/AI_City_project")
# Nếu shortcut được đặt ở vị trí/tên khác, thêm path chính xác vào danh sách tương ứng.
SOURCE_ROOT_FALLBACKS = []
PROJECT_ROOT_FALLBACKS = []

SOURCE_PARTS = [
    "1_20 - Quynh.v2i.yolo26",
    "21_40 - Vu.v3i.yolo26",
    "41_60 - Colabs.v1i.yolo26",
    "61_80 - Khoi.v2i.yolo26",
    "81_100 - Minh.v2i.yolo26",
]
VALIDATION_SEQUENCE_IDS = set(range(81, 101))
TARGET_NAMES = ["motorcycle", "rider_with_helmet", "rider_without_helmet"]
RUN_NAME = "yolo26s_baseline_seq81_100_v1"
DATASET_NAME = "aicity_3class_seq81_100_v1"


def _mounted_roots():
    drive_root = Path("/content/drive")
    roots = [drive_root / "MyDrive", drive_root / "Shared with me", drive_root / "Shareddrives"]
    return [root for root in roots if root.is_dir()]


def _discover_named_drive_folders(folder_name, max_depth=3):
    """Find same-named folders at shallow Drive depth, including Shared Drives."""
    found = set()
    for mount_root in _mounted_roots():
        queue = [(mount_root, 0)]
        visited = set()
        while queue:
            parent, depth = queue.pop(0)
            try:
                resolved_parent = str(parent.resolve())
                if resolved_parent in visited:
                    continue
                visited.add(resolved_parent)
                children = [child for child in parent.iterdir() if child.is_dir()]
            except (OSError, PermissionError):
                continue
            for child in children:
                if child.name == folder_name:
                    found.add(child)
                elif depth < max_depth:
                    queue.append((child, depth + 1))
    return found


def _is_source_root(path):
    return path.is_dir() and all(
        (path / part / "data.yaml").is_file()
        and (path / part / "train" / "images").is_dir()
        and (path / part / "train" / "labels").is_dir()
        for part in SOURCE_PARTS
    )


def _discover_source_roots(max_depth=4):
    """Find a parent containing all five parts, even if its shortcut has another name."""
    found = set()
    for mount_root in _mounted_roots():
        queue = [(mount_root, 0)]
        visited = set()
        while queue:
            parent, depth = queue.pop()
            try:
                key = str(parent.resolve())
                if key in visited:
                    continue
                visited.add(key)
                children = [child for child in parent.iterdir() if child.is_dir()]
            except OSError:
                continue
            for child in children:
                if _is_source_root(child):
                    found.add(child)
                elif depth < max_depth:
                    queue.append((child, depth + 1))
    return found

def _resolve_source_root(preferred, fallbacks):
    candidates = [Path(preferred), *(Path(p) for p in fallbacks)]
    candidates.extend([
        Path("/content/drive/MyDrive") / preferred.name,
        Path("/content/drive/MyDrive/Shared with me") / preferred.name,
        *_discover_named_drive_folders(preferred.name),
        *_discover_source_roots(),
    ])
    unique = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    valid = [path for path in unique if _is_source_root(path)]
    if Path(preferred) in valid:
        return Path(preferred)
    if len(valid) == 1:
        print(f"Tự nhận diện SOURCE_ROOT trong thư mục Drive được chia sẻ: {valid[0]}")
        return valid[0]
    if len(valid) > 1:
        raise RuntimeError(
            "Tìm thấy nhiều thư mục chứa đủ 5 phần dataset; hãy đặt SOURCE_ROOT chính xác "
            "hoặc chỉ giữ một path hợp lệ trong SOURCE_ROOT_FALLBACKS: "
            + ", ".join(map(str, valid))
        )
    raise FileNotFoundError(
        "Không tìm thấy thư mục có đủ 5 phần dataset. Drive có thể vẫn mount tài khoản trước, hoặc shortcut chưa nằm trong My Drive. "
        "Remount đúng tài khoản rồi Add shortcut to My Drive; nếu cần, thêm path vào SOURCE_ROOT_FALLBACKS. "
        f"Đã kiểm tra: {', '.join(map(str, unique))}"
    )


def _project_state_score(path):
    markers = [
        path / "metadata" / f"{RUN_NAME}_run_config.json",
        path / "runs" / RUN_NAME / "weights" / "last.pt",
        path / "runs" / RUN_NAME / "weights" / "best.pt",
        path / "datasets" / DATASET_NAME / "manifest.json",
    ]
    return sum(marker.is_file() for marker in markers)


def _discover_project_state_roots(max_depth=4):
    """Find a shared project by its run/checkpoint markers, independent of folder name."""
    found = set()
    for mount_root in _mounted_roots():
        queue = [(mount_root, 0)]
        visited = set()
        while queue:
            parent, depth = queue.pop()
            try:
                key = str(parent.resolve())
                if key in visited:
                    continue
                visited.add(key)
                children = [child for child in parent.iterdir() if child.is_dir()]
            except OSError:
                continue
            for child in children:
                if _project_state_score(child) > 0:
                    found.add(child)
                elif depth < max_depth:
                    queue.append((child, depth + 1))
    return found

def _resolve_project_root(preferred, fallbacks):
    candidates = [Path(preferred), *(Path(p) for p in fallbacks)]
    candidates.extend([
        Path("/content/drive/MyDrive") / preferred.name,
        Path("/content/drive/MyDrive/Shared with me") / preferred.name,
        *_discover_named_drive_folders(preferred.name),
        *_discover_project_state_roots(),
        *_discover_source_roots(),
    ])
    unique = []
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    existing = [path for path in unique if path.is_dir()]
    if not existing:
        raise FileNotFoundError(
            "Không tìm thấy PROJECT_ROOT đã chia sẻ. Notebook dừng thay vì tự tạo project trống làm mất dấu "
            "last.pt. Hãy Add shortcut thư mục AI_City_project vào My Drive hoặc thêm path đúng vào "
            f"PROJECT_ROOT_FALLBACKS. Đã kiểm tra: {', '.join(map(str, unique))}"
        )
    scored = [(path, _project_state_score(path)) for path in existing]
    best_score = max(score for _, score in scored)
    if best_score > 0:
        best = [path for path, score in scored if score == best_score]
        if Path(preferred) in best:
            return Path(preferred)
        if len(best) == 1:
            print(f"Tự nhận diện PROJECT_ROOT có metadata/checkpoint dùng chung: {best[0]}")
            return best[0]
        raise RuntimeError(
            "Tìm thấy nhiều project Drive có metadata/checkpoint; hãy đặt PROJECT_ROOT chính xác: "
            + ", ".join(map(str, best))
        )
    if Path(preferred) in existing:
        return Path(preferred)
    if len(existing) == 1:
        print(f"Tự nhận diện PROJECT_ROOT trong thư mục Drive được chia sẻ: {existing[0]}")
        return existing[0]
    raise RuntimeError(
        "Có nhiều thư mục project cùng tên nhưng chưa thấy metadata/checkpoint để chọn an toàn. "
        "Hãy đặt PROJECT_ROOT chính xác hoặc bổ sung PROJECT_ROOT_FALLBACKS: "
        + ", ".join(map(str, existing))
    )


SOURCE_ROOT = _resolve_source_root(SOURCE_ROOT, SOURCE_ROOT_FALLBACKS)
PROJECT_ROOT = _resolve_project_root(PROJECT_ROOT, PROJECT_ROOT_FALLBACKS)
print("SOURCE_ROOT:", SOURCE_ROOT.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())

MODEL_WEIGHTS = "yolo26s.pt"
TARGET_EPOCHS = 100
IMGSZ = 640
SEED = 42
BATCH = 16
WORKERS = 4
DEVICE = 0
OPTIMIZER = "AdamW"
LR0 = 0.001
LRF = 0.01
WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
PATIENCE = 100
CLOSE_MOSAIC = 10
AMP = True
DETERMINISTIC = True
SAVE_PERIOD = 1

# Confusion matrix dùng VAL_CONF/VAL_IOU; ảnh dự đoán dùng PRED_CONF/PRED_IOU.
VAL_CONF = 0.001
VAL_IOU = 0.45
PRED_CONF = 0.25
PRED_IOU = 0.45

DATASET_DIR = PROJECT_ROOT / "datasets" / DATASET_NAME
RUN_DIR = PROJECT_ROOT / "runs" / RUN_NAME
RUN_CONFIG_PATH = PROJECT_ROOT / "metadata" / f"{RUN_NAME}_run_config.json"
ENVIRONMENT_PATH = PROJECT_ROOT / "metadata" / f"{RUN_NAME}_environment.json"

if not str(PROJECT_ROOT.resolve()).startswith("/content/drive/"):
    raise ValueError("PROJECT_ROOT phải ở Google Drive, không được lưu chỉ trong /content.")
if not str(SOURCE_ROOT.resolve()).startswith("/content/drive/"):
    raise ValueError("SOURCE_ROOT phải trỏ tới dữ liệu đã đặt trên Google Drive.")



In [ ]:
import subprocess
# Cài Ultralytics đúng phiên bản đã lưu cho run; nếu là run mới thì cài bản hiện hành.
from pathlib import Path
ENVIRONMENT_PATH.parent.mkdir(parents=True, exist_ok=True)
if ENVIRONMENT_PATH.exists():
    saved = json.loads(ENVIRONMENT_PATH.read_text(encoding="utf-8"))
    version = saved.get("ultralytics_version")
    if not version:
        raise RuntimeError("Không có ultralytics_version trong environment log.")
    cmd = [sys.executable, "-m", "pip", "install", f"ultralytics=={version}", "pyyaml"]
    print("Cài Ultralytics đã lưu:", version)
else:
    cmd = [sys.executable, "-m", "pip", "install", "--upgrade", "ultralytics", "pyyaml"]
    print("Chưa có environment log; cài bản Ultralytics hiện hành.")
subprocess.check_call(cmd)


In [ ]:
import hashlib
import math
import re
from collections import Counter, defaultdict
from PIL import Image
import yaml
import pandas as pd
from IPython.display import display

_REQUIRED_CONFIG = ("DATASET_DIR", "SOURCE_ROOT", "SOURCE_PARTS", "TARGET_NAMES", "VALIDATION_SEQUENCE_IDS")
_missing_config = [name for name in _REQUIRED_CONFIG if name not in globals()]
if _missing_config:
    raise RuntimeError(
        "Thiếu biến cấu hình: " + ", ".join(_missing_config) + ". "
        "Hãy chạy cell cấu hình có drive.mount ở đầu notebook trước cell chuẩn bị dataset. "
        "Sau khi restart runtime, cần chạy lại các cell từ trên xuống."
    )

DATA_YAML = DATASET_DIR / "data.yaml"
MANIFEST_PATH = DATASET_DIR / "manifest.json"
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_ALIASES = {
    "motorcycle": 0, "motorbike": 0,
    "dhelmet": 1, "p1helmet": 1, "p2helmet": 1,
    "riderwithhelmet": 1, "helmetedrider": 1,
    "dnohelmet": 2, "p1nohelmet": 2, "p2nohelmet": 2,
    "riderwithouthelmet": 2, "nohelmetrider": 2,
}

def normalized_class_key(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

def destination_class(source_name, part_name):
    key = normalized_class_key(source_name)
    if key not in CLASS_ALIASES:
        raise ValueError(
            f"Không có mapping an toàn cho class {source_name!r} trong phần {part_name}; "
            "dừng trước khi tạo dataset."
        )
    return CLASS_ALIASES[key]

def names_from_yaml(data, part_name):
    if not isinstance(data, dict) or "names" not in data or "nc" not in data:
        raise ValueError(f"{part_name}: data.yaml thiếu nc hoặc names.")
    raw = data["names"]
    if isinstance(raw, list):
        names = [str(x) for x in raw]
    elif isinstance(raw, dict):
        pairs = sorted((int(k), str(v)) for k, v in raw.items())
        ids = [k for k, _ in pairs]
        if ids != list(range(len(ids))):
            raise ValueError(f"{part_name}: class ID không liên tục từ 0: {ids}")
        names = [v for _, v in pairs]
    else:
        raise ValueError(f"{part_name}: định dạng names không hỗ trợ.")
    if int(data["nc"]) != len(names):
        raise ValueError(f"{part_name}: nc không khớp độ dài names.")
    return names

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def parse_and_remap_label(path, source_names, part_name, source_project=None, raw_text=None):
    mapped, counts = [], Counter()
    rounding_tolerance_boxes = 0
    if raw_text is None:
        raw_text = path.read_text(encoding="utf-8")
    for line_number, raw in enumerate(raw_text.splitlines(), 1):
        fields = raw.strip().split()
        if not fields:
            continue
        if len(fields) != 5 or not re.fullmatch(r"\d+", fields[0]):
            raise ValueError(f"{path}:{line_number}: nhãn YOLO phải có class ID nguyên và 4 tọa độ.")
        source_id = int(fields[0])
        if source_id < 0 or source_id >= len(source_names):
            valid_names = ", ".join(f"{idx}:{name}" for idx, name in enumerate(source_names))
            yaml_path = SOURCE_ROOT / part_name / "data.yaml"
            raise ValueError(
                f"{path}:{line_number}: source class ID {source_id} is missing from {yaml_path}; "
                f"nc={len(source_names)}, declared classes=[{valid_names}], Roboflow project={source_project!r}. "
                "Stop and correct/confirm the source YAML; labels are never dropped or guessed."
            )
        target_id = destination_class(source_names[source_id], part_name)
        try:
            coords = [float(value) for value in fields[1:]]
        except ValueError as exc:
            raise ValueError(f"{path}:{line_number}: bbox có tọa độ không phải số.") from exc
        if not all(math.isfinite(v) for v in coords):
            raise ValueError(f"{path}:{line_number}: bbox có NaN/Inf.")
        x, y, w, h = coords
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"{path}:{line_number}: bbox xywh ngoài miền chuẩn hóa [0,1].")
        # Sai số biên chỉ cho phép làm tròn thập phân khi tính cạnh; giữ nguyên mọi tọa độ nguồn.
        eps = 1e-5
        left, top = x - w / 2, y - h / 2
        right, bottom = x + w / 2, y + h / 2
        overflow = max(0.0, -left, -top, right - 1.0, bottom - 1.0)
        if overflow > eps:
            raise ValueError(
                f"{path}:{line_number}: bbox vượt biên ảnh {overflow:.8g}, "
                f"vượt dung sai làm tròn {eps}."
            )
        if overflow > 0:
            rounding_tolerance_boxes += 1
        mapped.append(f"{target_id} " + " ".join(fields[1:]))  # bbox giữ nguyên
        counts[target_id] += 1
    return "\n".join(mapped) + ("\n" if mapped else ""), tuple(sorted(mapped)), counts, rounding_tolerance_boxes

def sequence_id(filename, part_name):
    match = re.match(r"^(\d{3})_", filename)
    if not match:
        raise ValueError(f"Không xác định được sequence từ {filename!r} trong {part_name}; dừng để tránh leakage.")
    value = int(match.group(1))
    if not 1 <= value <= 100:
        raise ValueError(f"Sequence ID ngoài 001..100: {filename!r}")
    return value

def fmt_groups(values):
    return ", ".join(f"{x:03d}" for x in sorted(values)) or "(không có)"

mapping_rows, source_summaries = [], []
source_project_by_part = {}
records_by_filename = {}
source_fingerprint = hashlib.sha256()
duplicate_copies = 0
groups_by_part = {}

for part_name in SOURCE_PARTS:
    root = SOURCE_ROOT / part_name
    yaml_path, image_dir, label_dir = root / "data.yaml", root / "train/images", root / "train/labels"
    if not yaml_path.is_file():
        raise FileNotFoundError(f"Thiếu YAML: {yaml_path}")
    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(f"{part_name}: cần train/images và train/labels.")
    source_data = yaml.safe_load(yaml_path.read_text(encoding="utf-8-sig"))
    source_names = names_from_yaml(source_data, part_name)
    source_project_by_part[part_name] = (source_data.get("roboflow") or {}).get("project")
    folder_range = re.match(r"^(\d{1,3}_\d{1,3})", part_name)
    project_range = re.match(r"^(\d{1,3}_\d{1,3})", str(source_project_by_part[part_name] or ""))
    if folder_range and project_range and folder_range.group(1) != project_range.group(1):
        print(f"WARNING: folder range {folder_range.group(1)} differs from YAML Roboflow project {source_project_by_part[part_name]!r}; class mapping still follows data.yaml.")
    for source_id, source_name in enumerate(source_names):
        target_id = destination_class(source_name, part_name)
        mapping_rows.append({
            "phần dữ liệu": part_name, "tên lớp nguồn": source_name, "ID nguồn": source_id,
            "tên lớp đích": TARGET_NAMES[target_id], "ID đích": target_id,
        })
    for readme_name in ("README.dataset.txt", "README.roboflow.txt"):
        readme = root / readme_name
        if readme.is_file():
            lines = readme.read_text(encoding="utf-8", errors="replace").splitlines()
            print(f"README {part_name}/{readme_name}: " + " | ".join(lines[:4])[:300])

    image_files = [p for p in image_dir.iterdir() if p.is_file()]
    unsupported_images = [p.name for p in image_files if p.suffix.lower() not in IMAGE_SUFFIXES]
    if unsupported_images:
        raise ValueError(f"{part_name}: định dạng ảnh không hỗ trợ, không bỏ qua: {unsupported_images[:5]}")
    images = sorted(image_files)
    non_txt_labels = [p.name for p in label_dir.iterdir() if p.is_file() and p.suffix.lower() != ".txt"]
    if non_txt_labels:
        raise ValueError(f"{part_name}: file không phải TXT trong labels, không bỏ qua: {non_txt_labels[:5]}")
    labels = sorted(p for p in label_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt")
    labels_by_stem = {p.stem: p for p in labels}
    if len(labels_by_stem) != len(labels):
        raise ValueError(f"{part_name}: trùng stem file nhãn.")
    stems = [p.stem for p in images]
    if len(stems) != len(set(stems)):
        raise ValueError(f"{part_name}: trùng stem ảnh.")
    images_by_stem = {p.stem: p for p in images}
    extra_labels = set(labels_by_stem) - set(stems)
    if extra_labels:
        raise ValueError(f"{part_name}: nhãn không có ảnh: {sorted(extra_labels)[:5]}")

    groups, part_boxes, empty_images = set(), Counter(), 0
    part_rounding_tolerance_boxes = 0
    part_verified_suffix_duplicates = 0
    part_exact_duplicates_skipped = 0
    for image_path in images:
        label_path = labels_by_stem.get(image_path.stem)
        duplicate_identity_name = image_path.name
        image_hash = None
        if label_path is None:
            suffix_match = re.fullmatch(r"(.+) \(\d+\)", image_path.stem)
            if suffix_match is None:
                raise ValueError(f"{part_name}: ảnh thiếu nhãn: {image_path.name}")
            original_stem = suffix_match.group(1)
            original_image = images_by_stem.get(original_stem)
            original_label = labels_by_stem.get(original_stem)
            if original_image is None or original_label is None:
                raise ValueError(
                    f"{part_name}: {image_path.name} có hậu tố bản sao nhưng không tìm thấy đủ "
                    f"ảnh và nhãn gốc {original_stem!r}; không đoán mapping nhãn."
                )
            image_hash = sha256_file(image_path)
            original_hash = sha256_file(original_image)
            if image_hash != original_hash:
                raise ValueError(
                    f"{part_name}: {image_path.name} thiếu nhãn và không giống byte ảnh gốc "
                    f"{original_image.name}; dừng, không mượn nhãn."
                )
            label_path = original_label
            duplicate_identity_name = original_image.name
            part_verified_suffix_duplicates += 1
        try:
            with Image.open(image_path) as im:
                im.verify()
        except Exception as exc:
            raise ValueError(f"Image is invalid or unreadable: {image_path}") from exc
        if image_hash is None:
            image_hash = sha256_file(image_path)
        raw_label_text = label_path.read_text(encoding="utf-8")
        raw_label_sha256 = hashlib.sha256(raw_label_text.encode("utf-8")).hexdigest()
        previous = records_by_filename.get(duplicate_identity_name)
        if (
            previous is not None
            and previous["image_sha256"] == image_hash
            and previous["source_label_sha256"] == raw_label_sha256
        ):
            # Exact copies reuse the annotation already remapped under the authoritative source YAML.
            duplicate_copies += 1
            part_exact_duplicates_skipped += 1
            continue
        group = sequence_id(image_path.name, part_name)
        groups.add(group)
        mapped_text, signature, box_counts, rounding_boxes = parse_and_remap_label(
            label_path, source_names, part_name, source_project_by_part.get(part_name), raw_label_text
        )
        part_rounding_tolerance_boxes += rounding_boxes
        if not signature:
            empty_images += 1
        part_boxes.update(box_counts)
        record = {
            "image_path": image_path, "filename": image_path.name, "stem": image_path.stem,
            "group": group, "mapped_text": mapped_text, "signature": signature,
            "box_counts": dict(box_counts), "image_sha256": image_hash,
            "source_label_sha256": raw_label_sha256, "first_part": part_name,
        }
        source_fingerprint.update(
            (part_name + "\t" + image_path.name + "\t" + image_hash + "\t" + repr(signature) + "\n").encode("utf-8")
        )
        if duplicate_identity_name in records_by_filename:
            old = records_by_filename[duplicate_identity_name]
            if old["image_sha256"] != image_hash or old["signature"] != signature:
                raise ValueError(
                    f"Tên trùng hoặc hậu tố bản sao đã xác minh nhưng ảnh/annotation sau remap khác: "
                    f"{old['first_part']} ({old['filename']}) và {part_name} ({image_path.name}). "
                    "Dừng, không bỏ annotation."
                )
            duplicate_copies += 1
        else:
            records_by_filename[duplicate_identity_name] = record
    groups_by_part[part_name] = groups
    source_summaries.append({
        "phần dữ liệu": part_name, "ảnh": len(images), "file nhãn": len(labels),
        "sequence": fmt_groups(groups), "ảnh không bbox": empty_images,
        "bbox nguồn": int(sum(part_boxes.values())),
        "exact duplicate copies skipped": part_exact_duplicates_skipped,
        "bbox sát biên do làm tròn (giữ nguyên)": part_rounding_tolerance_boxes,
        "ảnh hậu tố (n) xác minh trùng ảnh gốc": part_verified_suffix_duplicates,
    })

expected_ranges = [set(range(1,21)), set(range(21,41)), set(range(41,61)), set(range(61,81))]
for part_name, expected in zip(SOURCE_PARTS[:4], expected_ranges):
    if groups_by_part[part_name] != expected:
        raise ValueError(
            f"{part_name}: sequence thực tế không khớp tên thư mục; "
            f"thực tế={fmt_groups(groups_by_part[part_name])}; mong đợi={fmt_groups(expected)}."
        )

expected_validation_range = set(range(81, 101))
unexpected_validation_part_groups = groups_by_part[SOURCE_PARTS[4]] - expected_validation_range
if unexpected_validation_part_groups:
    raise ValueError(
        f"Unique samples in validation part {SOURCE_PARTS[4]} have sequence IDs outside 081..100: "
        f"{fmt_groups(unexpected_validation_part_groups)}; stop and inspect the source folder."
    )

all_records = list(records_by_filename.values())
all_groups = {r["group"] for r in all_records}
validation_groups = all_groups & VALIDATION_SEQUENCE_IDS
training_groups = all_groups - VALIDATION_SEQUENCE_IDS
if not training_groups or not validation_groups or training_groups & validation_groups:
    raise ValueError("Không có split train/validation hợp lệ theo sequence.")
hash_to_groups = defaultdict(set)
for record in all_records:
    hash_to_groups[record["image_sha256"]].add(record["group"])
if any(len(groups) > 1 for groups in hash_to_groups.values()):
    raise ValueError("Ảnh byte-identical xuất hiện ở nhiều sequence; không thể bảo đảm split sạch.")

display(pd.DataFrame(mapping_rows))
display(pd.DataFrame(source_summaries))
print(f"Ảnh duy nhất sau kiểm tra bản trùng: {len(all_records)}")
print(f"Exact duplicate copies collapsed only after image SHA256 and raw source-label verification: {duplicate_copies}")
print(f"Train sequences: {fmt_groups(training_groups)}")
print(f"Validation sequences: {fmt_groups(validation_groups)}")
print("Validation là holdout theo sequence, không gọi là test.")
print("Source fingerprint:", source_fingerprint.hexdigest())


In [ ]:
import shutil
from collections import Counter
import yaml

source_digest = source_fingerprint.hexdigest()
contract = {
    "dataset_version": 1,
    "source_root": SOURCE_ROOT.name,
    "source_parts": SOURCE_PARTS,
    "source_fingerprint": source_digest,
    "target_names": TARGET_NAMES,
    "validation_sequence_ids": sorted(VALIDATION_SEQUENCE_IDS),
    "training_sequence_ids": sorted(training_groups),
    "actual_validation_sequence_ids": sorted(validation_groups),
    "duplicate_filenames_verified_and_collapsed": duplicate_copies,
    "image_count": len(all_records),
}

if DATASET_DIR.exists():
    if not MANIFEST_PATH.is_file() or not DATA_YAML.is_file():
        raise FileExistsError(f"{DATASET_DIR} thiếu manifest/data.yaml; không ghi đè.")
    cached = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
    for key, expected in contract.items():`n        if key == "source_root":  # The shared folder mount path can differ between accounts.`n            continue
        if cached.get(key) != expected:
            raise RuntimeError(f"Dataset cache không khớp trường {key}; không tái sử dụng hoặc ghi đè.")
    print("Tái sử dụng dataset chuẩn hóa trên Drive:", DATASET_DIR)
else:
    DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    stage = DATASET_DIR.with_name(DATASET_DIR.name + ".building")
    if stage.exists():
        raise FileExistsError(f"Staging cũ còn tồn tại: {stage}; kiểm tra trước, notebook không tự xóa.")
    stage.mkdir(parents=True)
    try:
        for split in ("train", "val"):
            (stage / split / "images").mkdir(parents=True)
            (stage / split / "labels").mkdir(parents=True)
        counts = {
            split: {"images": 0, "boxes": Counter(), "images_per_class": Counter(), "empty": 0}
            for split in ("train", "val")
        }
        for record in all_records:
            split = "val" if record["group"] in validation_groups else "train"
            image_out = stage / split / "images" / record["filename"]
            label_out = stage / split / "labels" / f"{record['stem']}.txt"
            if image_out.exists() or label_out.exists():
                raise FileExistsError(f"Trùng tên file đầu ra: {record['filename']}")
            shutil.copy2(record["image_path"], image_out)
            label_out.write_text(record["mapped_text"], encoding="utf-8")
            counts[split]["images"] += 1
            counts[split]["boxes"].update(record["box_counts"])
            counts[split]["images_per_class"].update({k: 1 for k in record["box_counts"]})
            counts[split]["empty"] += int(not record["signature"])

        payload = {
            "path": str(DATASET_DIR.resolve()),
            "train": "train/images",
            "val": "val/images",
            "nc": 3,
            "names": TARGET_NAMES,
        }
        (stage / "data.yaml").write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding="utf-8")
        manifest = {
            **contract,
            "split": {
                "method": "sequence ID from image filename prefix ^(\\d{3})_; no random frame split",
                "train_sequence_ids": sorted(training_groups),
                "validation_sequence_ids": sorted(validation_groups),
                "validation_is_test": False,
            },
            "source_summaries": source_summaries,
            "class_mapping": mapping_rows,
            "class_counts": {
                split: {
                    "images": value["images"],
                    "bbox_per_class": {str(i): int(value["boxes"].get(i, 0)) for i in range(3)},
                    "images_per_class": {str(i): int(value["images_per_class"].get(i, 0)) for i in range(3)},
                    "images_without_boxes": value["empty"],
                }
                for split, value in counts.items()
            },
        }
        (stage / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        for split in ("train", "val"):
            out_images = list((stage / split / "images").iterdir())
            out_labels = list((stage / split / "labels").glob("*.txt"))
            if len(out_images) != len(out_labels) or len({p.name for p in out_images}) != len(out_images):
                raise AssertionError(f"{split}: ảnh/nhãn thiếu hoặc tên file trùng.")
            for label in out_labels:
                for row, line in enumerate(label.read_text(encoding="utf-8").splitlines(), 1):
                    if line.strip() and (len(line.split()) != 5 or int(line.split()[0]) not in (0,1,2)):
                        raise AssertionError(f"Nhãn đích lỗi: {label}:{row}")
        shutil.move(str(stage), str(DATASET_DIR))
    except Exception:
        if stage.exists():
            shutil.rmtree(stage)
        raise

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("Normalized dataset:", DATASET_DIR)
print("data.yaml:", DATA_YAML)
print("Train sequences:", fmt_groups(manifest["split"]["train_sequence_ids"]))
print("Validation sequences:", fmt_groups(manifest["split"]["validation_sequence_ids"]))
print("Validation không phải test.")
display(pd.DataFrame([
    {
        "split": split,
        "images": value["images"],
        "class_id": class_id,
        "class_name": TARGET_NAMES[class_id],
        "images_with_class": value["images_per_class"].get(str(class_id), 0),
        "bounding_boxes": value["bbox_per_class"].get(str(class_id), 0),
        "images_without_boxes": value["images_without_boxes"],
    }
    for split, value in manifest["class_counts"].items()
    for class_id in range(3)
]))



In [ ]:
import datetime as dt
import json
import os
import platform
import sys
import time
import torch
import ultralytics
import yaml
from ultralytics import YOLO

RUNS_ROOT = PROJECT_ROOT / "runs"
LOCK_ROOT = PROJECT_ROOT / "locks"
LOG_ROOT = PROJECT_ROOT / "logs"
METADATA_ROOT = PROJECT_ROOT / "metadata"
for folder in (RUNS_ROOT, LOCK_ROOT, LOG_ROOT, METADATA_ROOT):
    folder.mkdir(parents=True, exist_ok=True)
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU CUDA; chọn GPU runtime trong Colab.")
GPU_NAME = torch.cuda.get_device_name(0)
ULTRALYTICS_VERSION = ultralytics.__version__
TORCH_VERSION = torch.__version__
DATA_YAML_SHA256 = sha256_file(DATA_YAML)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

TRAIN_ARGS = {
    "data": str(DATA_YAML.resolve()), "epochs": TARGET_EPOCHS, "imgsz": IMGSZ,
    "seed": SEED, "batch": BATCH, "workers": WORKERS, "device": DEVICE,
    "optimizer": OPTIMIZER, "lr0": LR0, "lrf": LRF, "weight_decay": WEIGHT_DECAY,
    "warmup_epochs": WARMUP_EPOCHS, "patience": PATIENCE, "close_mosaic": CLOSE_MOSAIC,
    "amp": AMP, "deterministic": DETERMINISTIC, "save": True, "save_period": SAVE_PERIOD,
    "cache": False, "project": str(RUNS_ROOT), "name": RUN_NAME, "exist_ok": False,
}
RUN_SIGNATURE = {
    "experiment": "YOLO26s_baseline", "model": MODEL_WEIGHTS, "train_args": TRAIN_ARGS,
    "dataset_source_fingerprint": manifest["source_fingerprint"],
    "normalized_data_yaml_sha256": DATA_YAML_SHA256,
    "target_names": TARGET_NAMES,
    "train_sequence_ids": manifest["split"]["train_sequence_ids"],
    "validation_sequence_ids": manifest["split"]["validation_sequence_ids"],
    "ultralytics_version": ULTRALYTICS_VERSION,
    "torch_version_at_start": TORCH_VERSION,
}
ENV_RECORD = {
    "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "python": sys.version, "platform": platform.platform(),
    "ultralytics_version": ULTRALYTICS_VERSION, "torch_version": TORCH_VERSION,
    "cuda_version": torch.version.cuda, "gpu": GPU_NAME, "seed": SEED,
    "optimizer": OPTIMIZER, "batch": BATCH, "imgsz": IMGSZ,
    "epochs_total": TARGET_EPOCHS, "learning_rate": {"lr0": LR0, "lrf": LRF},
    "weight_decay": WEIGHT_DECAY, "data_yaml": str(DATA_YAML.resolve()),
    "data_yaml_sha256": DATA_YAML_SHA256, "checkpoint_dir": str(RUN_DIR / "weights"),
}
if ENVIRONMENT_PATH.exists():
    prior_env = json.loads(ENVIRONMENT_PATH.read_text(encoding="utf-8"))
    if prior_env.get("ultralytics_version") != ULTRALYTICS_VERSION:
        raise RuntimeError("Ultralytics hiện tại khác phiên bản run; không resume.")
else:
    ENVIRONMENT_PATH.write_text(json.dumps(ENV_RECORD, ensure_ascii=False, indent=2), encoding="utf-8")

LOCK_PATH = LOCK_ROOT / f"{RUN_NAME}.lock"
try:
    lock_fd = os.open(str(LOCK_PATH), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError as exc:
    raise RuntimeError(f"Run đang có lock {LOCK_PATH}; không chạy đồng thời.") from exc
with os.fdopen(lock_fd, "w", encoding="utf-8") as stream:
    json.dump({"created_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "run": RUN_NAME}, stream)

LAST_CKPT = RUN_DIR / "weights" / "last.pt"
BEST_CKPT = RUN_DIR / "weights" / "best.pt"
ARGS_YAML = RUN_DIR / "args.yaml"
RESULTS_CSV = RUN_DIR / "results.csv"
CONSOLE_LOG = LOG_ROOT / f"{RUN_NAME}_console.log"
EVENT_LOG = LOG_ROOT / f"{RUN_NAME}_events.jsonl"
SESSION_START = time.monotonic()

class Tee:
    def __init__(self, original, logfile):
        self.original, self.logfile = original, logfile
    def write(self, value):
        self.original.write(value)
        self.logfile.write(value)
        self.logfile.flush()
        return len(value)
    def flush(self):
        self.original.flush()
        self.logfile.flush()
    def isatty(self):
        return self.original.isatty() if hasattr(self.original, "isatty") else False
    @property
    def encoding(self):
        return getattr(self.original, "encoding", "utf-8")

def event(name, **values):
    row = {"event": name, "time_utc": dt.datetime.now(dt.timezone.utc).isoformat(), **values}
    with EVENT_LOG.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

def on_train_start(trainer):
    args = getattr(trainer, "args", None)
    opt = getattr(trainer, "optimizer", None)
    event(
        "train_start",
        optimizer_actual_class=type(opt).__name__ if opt is not None else None,
        batch=getattr(args, "batch", None), imgsz=getattr(args, "imgsz", None),
        epochs=getattr(args, "epochs", None), seed=getattr(args, "seed", None),
        gpu=GPU_NAME, ultralytics=ULTRALYTICS_VERSION, torch=TORCH_VERSION,
    )

def on_epoch_end(trainer):
    event(
        "epoch_end", epoch_completed=int(getattr(trainer, "epoch", -1)) + 1,
        target_epochs=TARGET_EPOCHS, elapsed_seconds=round(time.monotonic() - SESSION_START, 2),
    )

def on_train_end(trainer):
    event("train_end", elapsed_seconds=round(time.monotonic() - SESSION_START, 2))

def validate_run_config():
    if not RUN_CONFIG_PATH.is_file():
        raise RuntimeError(f"Thiếu run config {RUN_CONFIG_PATH}; không resume.")
    if json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8")) != RUN_SIGNATURE:
        raise RuntimeError("Cấu hình/dataset/split/phiên bản không khớp checkpoint; không resume.")
    if not ARGS_YAML.is_file():
        raise RuntimeError(f"Checkpoint có nhưng thiếu args.yaml: {ARGS_YAML}")
    args = yaml.safe_load(ARGS_YAML.read_text(encoding="utf-8"))
    expected = {
        "epochs": TARGET_EPOCHS, "imgsz": IMGSZ, "seed": SEED, "batch": BATCH,
        "optimizer": OPTIMIZER, "data": str(DATA_YAML.resolve()),
    }
    for key, value in expected.items():
        actual = args.get(key)
        if key == "data" and actual:
            actual = str(Path(actual).resolve())
        if actual != value:
            raise RuntimeError(f"args.yaml không khớp {key}: {actual!r} != {value!r}")
    if Path(str(args.get("model", ""))).name != MODEL_WEIGHTS:
        raise RuntimeError("args.yaml không xác nhận model YOLO26s baseline.")

def completed_epochs(path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    done = int(checkpoint.get("epoch", -1)) + 1
    if not 0 <= done <= TARGET_EPOCHS:
        raise RuntimeError(f"Epoch trong checkpoint không hợp lệ: {done}")
    return done

event(
    "session_environment", gpu=GPU_NAME, torch=TORCH_VERSION, ultralytics=ULTRALYTICS_VERSION,
    seed=SEED, optimizer=OPTIMIZER, batch=BATCH, imgsz=IMGSZ,
    data_yaml=str(DATA_YAML), run_dir=str(RUN_DIR),
)
try:
    if LAST_CKPT.is_file():
        validate_run_config()
        done = completed_epochs(LAST_CKPT)
        print(f"Đã xong {done}/{TARGET_EPOCHS} epoch.")
        if done == TARGET_EPOCHS:
            if not BEST_CKPT.is_file():
                raise FileNotFoundError(f"Đủ epoch nhưng thiếu best.pt: {BEST_CKPT}")
            print("Đã đủ 100 epoch; không train lại.")
        else:
            model = YOLO(str(LAST_CKPT))
            model.add_callback("on_train_start", on_train_start)
            model.add_callback("on_train_epoch_end", on_epoch_end)
            model.add_callback("on_train_end", on_train_end)
            print(f"Resume từ last.pt với resume=True; tổng mục tiêu vẫn là {TARGET_EPOCHS}.")
            with CONSOLE_LOG.open("a", encoding="utf-8") as logfile:
                old_out, old_err = sys.stdout, sys.stderr
                sys.stdout, sys.stderr = Tee(old_out, logfile), Tee(old_err, logfile)
                try:
                    model.train(resume=True)
                finally:
                    sys.stdout, sys.stderr = old_out, old_err
    else:
        if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
            raise RuntimeError(f"Run directory có nội dung nhưng thiếu last.pt: {RUN_DIR}; không ghi đè.")
        if RUN_CONFIG_PATH.exists():
            raise RuntimeError(f"Run config đã có nhưng thiếu last.pt: {RUN_CONFIG_PATH}; không tạo run mới.")
        print(f"Không tìm thấy checkpoint tại {LAST_CKPT}; bắt đầu run mới YOLO26s baseline.")
        try:
            model = YOLO(MODEL_WEIGHTS)
        except Exception as exc:
            raise RuntimeError(
                f"Ultralytics {ULTRALYTICS_VERSION} cannot load requested {MODEL_WEIGHTS}; "
                "stop without switching models."
            ) from exc
        RUN_CONFIG_PATH.write_text(json.dumps(RUN_SIGNATURE, ensure_ascii=False, indent=2), encoding="utf-8")
        model.add_callback("on_train_start", on_train_start)
        model.add_callback("on_train_epoch_end", on_epoch_end)
        model.add_callback("on_train_end", on_train_end)
        with CONSOLE_LOG.open("a", encoding="utf-8") as logfile:
            old_out, old_err = sys.stdout, sys.stderr
            sys.stdout, sys.stderr = Tee(old_out, logfile), Tee(old_err, logfile)
            try:
                model.train(**TRAIN_ARGS)
            finally:
                sys.stdout, sys.stderr = old_out, old_err
    if LAST_CKPT.is_file():
        print(f"Checkpoint hiện có: {completed_epochs(LAST_CKPT)}/{TARGET_EPOCHS} epoch.")
    event(
        "session_finished", elapsed_seconds=round(time.monotonic() - SESSION_START, 2),
        last_checkpoint=str(LAST_CKPT), best_checkpoint=str(BEST_CKPT),
        results_csv=str(RESULTS_CSV), args_yaml=str(ARGS_YAML),
    )
finally:
    if LOCK_PATH.exists():
        LOCK_PATH.unlink()


In [ ]:
import datetime as dt
import inspect
import json
import numpy as np
import pandas as pd
import re
import torch
from ultralytics import YOLO
from ultralytics.models.yolo.detect.val import DetectionValidator

if not BEST_CKPT.is_file():
    raise FileNotFoundError(f"Chưa có best.pt: {BEST_CKPT}")
if not ARGS_YAML.is_file() or not RESULTS_CSV.is_file():
    raise FileNotFoundError("Thiếu args.yaml hoặc results.csv; kiểm tra run trước.")

best_model = YOLO(str(BEST_CKPT))
captured = []

class CapturingDetectionValidator(DetectionValidator):
    def __call__(self, *args, **kwargs):
        result = super().__call__(*args, **kwargs)
        captured.append(self)
        return result

stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
eval_project = PROJECT_ROOT / "evaluation"
eval_name = f"best_validation_{stamp}"
eval_dir = eval_project / eval_name
print(f"Đánh giá best.pt trên validation: conf={VAL_CONF}, IoU={VAL_IOU}; không đọc test.")
metrics = best_model.val(
    data=str(DATA_YAML), split="val", conf=VAL_CONF, iou=VAL_IOU, plots=True,
    project=str(eval_project), name=eval_name, exist_ok=False,
    validator=CapturingDetectionValidator,
)
box = metrics.box
ap_class_ids = [int(v) for v in np.asarray(getattr(box, "ap_class_index", [])).tolist()]

def class_values(key):
    values = getattr(box, key, None)
    if values is None:
        return {}
    arr = np.asarray(values)
    if arr.ndim == 0 or len(arr) != len(ap_class_ids):
        return {}
    return {class_id: float(arr[i]) for i, class_id in enumerate(ap_class_ids)}

per_class = pd.DataFrame([
    {
        "class_id": i, "class_name": name,
        "precision": class_values("p").get(i), "recall": class_values("r").get(i),
        "AP50": class_values("ap50").get(i), "AP50_95": class_values("ap").get(i),
    }
    for i, name in enumerate(TARGET_NAMES)
])
per_class.to_csv(eval_dir / "per_class_metrics.csv", index=False)
summary = {
    "precision": float(box.mp), "recall": float(box.mr),
    "mAP50": float(box.map50), "mAP50_95": float(box.map),
    "validation_confidence": VAL_CONF, "validation_iou": VAL_IOU,
    "best_checkpoint": str(BEST_CKPT), "data_yaml": str(DATA_YAML),
    "ultralytics_version": ULTRALYTICS_VERSION, "torch_version": torch.__version__,
    "gpu": torch.cuda.get_device_name(0), "seed": SEED, "optimizer": OPTIMIZER,
    "batch": BATCH, "imgsz": IMGSZ, "epochs_target": TARGET_EPOCHS,
    "args_yaml": str(ARGS_YAML), "results_csv": str(RESULTS_CSV),
}
(eval_dir / "metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([summary]))
display(per_class)
print("AP và Recall rider_without_helmet:")
print(per_class.loc[per_class["class_name"] == "rider_without_helmet"].to_string(index=False))

validator = captured[-1] if captured else None
confusion = getattr(validator, "confusion_matrix", None) if validator else None
matrix = getattr(confusion, "matrix", None) if confusion is not None else None
if matrix is None:
    print("Validator không trả ma trận số; ảnh confusion matrix do plots=True nằm trong", eval_dir)
else:
    matrix = np.asarray(matrix)
    nc = len(TARGET_NAMES)
    labels = TARGET_NAMES + ["background"]
    if matrix.shape != (nc + 1, nc + 1):
        print(f"Shape confusion matrix {matrix.shape}; không tự diễn giải.")
    else:
        pd.DataFrame(matrix, index=labels, columns=labels).to_csv(eval_dir / "confusion_matrix.csv")
        source = inspect.getsource(type(confusion).process_batch)
        compact = re.sub(r"\s+", "", source)
        true_rows = "self.matrix[gt_classes[m0].int(),detection_classes[m1].int()]" in compact
        pred_rows = "self.matrix[detection_classes[m1].int(),gt_classes[m0].int()]" in compact
        (eval_dir / "confusion_matrix_process_batch_source.txt").write_text(source, encoding="utf-8")
        class_id = 2
        if true_rows:
            tp = int(matrix[class_id, class_id])
            fn = int(matrix[class_id, :].sum() - tp)
            fp = int(matrix[:, class_id].sum() - tp)
            axis = "rows=true, final row=background; columns=prediction, final column=background"
        elif pred_rows:
            tp = int(matrix[class_id, class_id])
            fn = int(matrix[:, class_id].sum() - tp)
            fp = int(matrix[class_id, :].sum() - tp)
            axis = "rows=prediction, final row=background; columns=true, final column=background"
        else:
            tp = fn = fp = None
            axis = "unknown; TP/FN/FP not calculated"
        if tp is not None:
            counts = {
                "class_name": TARGET_NAMES[class_id], "TP": tp, "FN": fn, "FP": fp,
                "axis": axis, "confidence": VAL_CONF, "iou": VAL_IOU,
                "matching": "Ultralytics ConfusionMatrix.process_batch, class-aware IoU matching",
            }
            (eval_dir / "rider_without_helmet_TP_FN_FP.json").write_text(
                json.dumps(counts, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            print("Confusion matrix:", axis, "| counts:", counts)
        else:
            print("Đã lưu ma trận và source process_batch; không xác định chắc chiều trục nên không tính TP/FN/FP.")
print(f"Confusion matrix dùng conf={VAL_CONF}, IoU={VAL_IOU}; plots: {eval_dir}")

pred_dir = PROJECT_ROOT / "predictions"
pred_name = f"validation_best_conf{str(PRED_CONF).replace('.', 'p')}_{stamp}"
best_model.predict(
    source=str(DATASET_DIR / "val" / "images"), save=True,
    conf=PRED_CONF, iou=PRED_IOU, project=str(pred_dir), name=pred_name,
    exist_ok=False, verbose=False,
)
print(f"Ảnh dự đoán validation (conf={PRED_CONF}, IoU={PRED_IOU}): {pred_dir / pred_name}")


## Kết quả và điều kiện so sánh SCConv

Run baseline dùng split và cấu hình được lưu trong manifest.json, metadata run config, args.yaml, results.csv, logs và environment.json trên Drive. Đánh giá ở trên chỉ dùng validation.

Lượt SCConv sau phải tạo run riêng, dùng cùng data.yaml, split, seed, imgsz, batch và hyperparameter đã lưu. Không resume từ last.pt baseline; không thêm/thay module nếu chưa có đúng mã C3k2_SCConv của dự án.
